# Evaluasi dan Export Artefak

Notebook ini menghitung evaluasi untuk seluruh model klasik dan CNN, membuat visualisasi evaluasi, memilih model final, lalu menyimpan metrics dan model terbaik.

## 1. Fungsi Evaluasi

Confusion matrix, ROC curve, classification report, training curve, dan tabel ringkasan dibuat terpusat agar notebook modeling tetap fokus pada training model.

In [ ]:

def plot_confusion_matrix(matrix: np.ndarray, scenario_id: str) -> None:
    fig, ax = plt.subplots(figsize=(5, 5))
    display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=CLASS_NAMES)
    display.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    plt.title(f'Confusion Matrix - {scenario_id}')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'confusion_matrix_{scenario_id}.png', dpi=180)
    plt.show()


def plot_roc_curve(y_true: np.ndarray, y_score: np.ndarray, scenario_id: str) -> float:
    plt.figure(figsize=(5.5, 5))
    if len(np.unique(y_true)) < 2:
        roc_auc = float('nan')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='AUC = nan')
        plt.text(0.5, 0.5, 'ROC tidak dihitung\nkarena hanya ada satu kelas', ha='center', va='center')
    else:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {scenario_id}')
    plt.grid(alpha=0.3)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'roc_curve_{scenario_id}.png', dpi=180)
    plt.show()
    return float(roc_auc)


def plot_training_history(history: dict, scenario_id: str) -> None:
    plt.figure(figsize=(11, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.get('accuracy', []), marker='o', label='train')
    plt.plot(history.get('val_accuracy', []), marker='o', label='validation')
    plt.title(f'Accuracy - {scenario_id}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.get('loss', []), marker='o', label='train')
    plt.plot(history.get('val_loss', []), marker='o', label='validation')
    plt.title(f'Loss - {scenario_id}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'training_history_{scenario_id}.png', dpi=180)
    plt.show()


def evaluate_predictions(
    scenario_id: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: np.ndarray,
    model_type: str,
    config: dict | None = None,
    history: dict | None = None,
    test_loss: float | None = None,
    extra: dict | None = None,
) -> dict:
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    roc_auc = plot_roc_curve(y_true, y_score, scenario_id)

    result = {
        'scenario': scenario_id,
        'model_type': model_type,
        'config': config or {},
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_true, y_pred, zero_division=0)),
        'auc': roc_auc,
        'classification_report': report_dict,
        'confusion_matrix': matrix.tolist(),
    }
    if history is not None:
        result['history'] = {key: [float(v) for v in values] for key, values in history.items()}
    if test_loss is not None:
        result['test_loss'] = float(test_loss)
    if extra:
        result.update(extra)

    print(f'\n=== {scenario_id} ===')
    print(f"Accuracy : {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall   : {result['recall']:.4f}")
    print(f"F1-score : {result['f1_score']:.4f}")
    print(f"AUC      : {result['auc']:.4f}")
    print(classification_report(y_true, y_pred, labels=[0, 1], target_names=CLASS_NAMES, zero_division=0))
    plot_confusion_matrix(matrix, scenario_id)
    if history is not None:
        plot_training_history(history, scenario_id)
    return result


## 2. Evaluasi Model Klasik

Model SVM yang sudah dilatih dievaluasi pada test set menggunakan HOG feature extraction yang sama dengan tahap training.

In [ ]:

scenario_results = []

for experiment in classical_experiments:
    scenario_id = experiment['scenario_id']
    scenario_name = experiment['scenario_name']
    model = classical_models[scenario_id]

    X_test_img, y_test = load_images_for_classical(scenario_name, 'test')
    X_test = extract_hog_features(X_test_img)
    y_score = model.decision_function(X_test)
    y_pred = (y_score >= 0).astype(np.int64)

    scenario_results.append(
        evaluate_predictions(
            scenario_id,
            y_test,
            y_pred,
            y_score,
            model_type='HOG + LinearSVC',
            config=experiment['config'],
            extra={
                'validation_accuracy': classical_feature_sets[scenario_id]['validation_accuracy'],
                'model_path': classical_feature_sets[scenario_id].get('model_path'),
            },
        )
    )


## 3. Evaluasi Model CNN

Setiap model CNN dievaluasi pada test set sesuai skenario dataset-nya. Prediksi, metric Keras, training history, dan checkpoint dicatat pada hasil evaluasi.

In [ ]:

for scenario_id, record in cnn_training_records.items():
    model = trained_models[scenario_id]
    config = record['config']
    scenario_name = record['scenario_name']
    batch_size = int(config.get('batch_size', BATCH_SIZE))
    test_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'test', shuffle=False, batch_size=batch_size))

    test_metrics = model.evaluate(test_ds, verbose=0)
    metric_names = model.metrics_names
    test_metric_dict = {name: float(value) for name, value in zip(metric_names, test_metrics)}

    y_true = []
    y_score = []
    for images, labels in test_ds:
        scores = model.predict(images, verbose=0).reshape(-1)
        y_true.extend(labels.numpy().reshape(-1).astype(np.int64).tolist())
        y_score.extend(scores.tolist())

    y_true = np.array(y_true, dtype=np.int64)
    y_score = np.array(y_score, dtype=np.float32)
    y_pred = (y_score >= 0.5).astype(np.int64)

    result = evaluate_predictions(
        scenario_id,
        y_true,
        y_pred,
        y_score,
        model_type='Custom CNN Conv2D from scratch',
        config=config,
        history=record['history'],
        test_loss=test_metric_dict.get('loss'),
        extra={
            'keras_test_metrics': test_metric_dict,
            'checkpoint_path': record['checkpoint_path'],
            'epochs_trained': record['epochs_trained'],
            'best_val_loss': record['best_val_loss'],
            'best_val_accuracy': record['best_val_accuracy'],
            'purpose': record.get('purpose', ''),
        },
    )
    scenario_results.append(result)


## Sampel Prediksi dengan Bounding Box

Dua contoh gambar dari test set ditampilkan untuk setiap model: satu `with_mask` dan satu `without_mask`. Gambar asli diberi bounding box wajah terbesar, lalu judul panel menampilkan label asli dan hasil prediksi model agar keluaran tiap skenario dapat dibandingkan secara visual.


In [ ]:
def detect_largest_face_box(image_rgb: np.ndarray, detector: cv2.CascadeClassifier) -> tuple[tuple[int, int, int, int], bool]:
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(35, 35))
    if len(faces) == 0:
        return (0, 0, image_rgb.shape[1], image_rgb.shape[0]), False
    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    return (int(x), int(y), int(w), int(h)), True


def draw_prediction_bbox(image_rgb: np.ndarray, box: tuple[int, int, int, int], label: str, detected: bool) -> np.ndarray:
    canvas = image_rgb.copy()
    x, y, w, h = box
    color = (39, 174, 96) if label == 'with_mask' else (235, 87, 87)
    line_width = max(2, min(4, image_rgb.shape[1] // 180))
    cv2.rectangle(canvas, (x, y), (x + w, y + h), color, line_width)

    # Fallback full-image box is only a visual boundary; avoid long text that covers the face.
    text = label if detected else f'{label} (fallback)'
    font_scale = max(0.35, min(0.65, image_rgb.shape[1] / 720))
    thickness = max(1, min(2, image_rgb.shape[1] // 360))
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)

    text_x = max(2, min(x, canvas.shape[1] - tw - 8))
    if y - th - baseline - 8 >= 0:
        text_y = y - 6
        bg_y1 = text_y - th - baseline - 4
        bg_y2 = text_y + baseline + 2
    else:
        text_y = min(canvas.shape[0] - baseline - 4, y + h + th + 8)
        bg_y1 = text_y - th - baseline - 4
        bg_y2 = text_y + baseline + 2

    cv2.rectangle(canvas, (text_x, max(0, bg_y1)), (min(text_x + tw + 8, canvas.shape[1]), min(bg_y2, canvas.shape[0])), color, -1)
    cv2.putText(canvas, text, (text_x + 4, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
    return canvas


def get_test_sample_by_class(class_name: str) -> Path:
    fallback_path = None
    for image_path, label in splits['test']:
        if label != class_name:
            continue
        if fallback_path is None:
            fallback_path = image_path
        image_rgb = load_original_rgb(image_path)
        _, detected = detect_largest_face_box(image_rgb, sample_bbox_detector)
        if detected:
            return image_path
    if fallback_path is None:
        raise ValueError(f'Tidak ada sampel test untuk class {class_name}')
    return fallback_path


def predict_sample_for_experiment(experiment: dict, image_path: Path, model_kind: str) -> str:
    scenario_id = experiment['scenario_id']
    scenario_name = experiment['scenario_name']
    use_enhancement = 'enhanced' in scenario_name
    use_face_crop = 'crop' in scenario_name
    processed, _ = preprocess_image(
        image_path,
        use_enhancement=use_enhancement,
        use_face_crop=use_face_crop,
        detector=sample_bbox_detector,
    )
    if processed is None:
        return 'unreadable'

    if model_kind == 'classical':
        features = extract_hog_features(np.array([processed], dtype=np.uint8))
        score = float(classical_models[scenario_id].decision_function(features).reshape(-1)[0])
        pred_idx = int(score >= 0.0)
    else:
        model_input = np.expand_dims(processed.astype(np.float32) / 255.0, axis=0)
        score = float(trained_models[scenario_id].predict(model_input, verbose=0).reshape(-1)[0])
        pred_idx = int(score >= 0.5)
    return CLASS_NAMES[pred_idx]


def short_scenario_name(scenario_id: str) -> str:
    return scenario_id.replace('_cnn_', '_cnn\n').replace('_hog_', '_hog\n')


def plot_sample_predictions_with_bbox() -> None:
    sample_paths = {class_name: get_test_sample_by_class(class_name) for class_name in CLASS_NAMES}
    print('Sample paths used for bbox visualization:')
    for class_name, image_path in sample_paths.items():
        print(f'- {class_name}: {image_path}')

    experiments = [(experiment, 'classical') for experiment in classical_experiments]
    experiments.extend((experiment, 'cnn') for experiment in cnn_experiments)

    fig, axes = plt.subplots(len(experiments), len(CLASS_NAMES), figsize=(11, max(3, len(experiments) * 2.7)))
    if len(experiments) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (experiment, model_kind) in enumerate(experiments):
        for col_idx, class_name in enumerate(CLASS_NAMES):
            image_path = sample_paths[class_name]
            original = load_original_rgb(image_path)
            pred_label = predict_sample_for_experiment(experiment, image_path, model_kind)
            box, detected = detect_largest_face_box(original, sample_bbox_detector)
            visual = draw_prediction_bbox(original, box, pred_label, detected)

            ax = axes[row_idx, col_idx]
            ax.imshow(visual)
            ax.set_title(f"{short_scenario_name(experiment['scenario_id'])}\ntrue: {class_name} | pred: {pred_label}", fontsize=8)
            ax.axis('off')

    plt.tight_layout(pad=1.1)
    output_path = OUTPUT_DIR / 'sample_predictions_with_bbox_by_model.png'
    plt.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved sample prediction grid: {output_path}')


sample_bbox_detector = cv2.CascadeClassifier(str(Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'))
plot_sample_predictions_with_bbox()


## 4. Perbandingan Semua Skenario

Seluruh hasil klasik dan CNN diringkas dalam tabel dan grafik perbandingan agar scenario terbaik terlihat jelas.

In [ ]:

def plot_scenario_comparison(results: list[dict]) -> None:
    names = [result['scenario'] for result in results]
    accuracy = [result['accuracy'] for result in results]
    precision = [result['precision'] for result in results]
    recall = [result['recall'] for result in results]
    f1 = [result['f1_score'] for result in results]
    auc_scores = [result['auc'] for result in results]

    x = np.arange(len(names))
    width = 0.16
    plt.figure(figsize=(16, 6))
    plt.bar(x - 2 * width, accuracy, width, label='accuracy')
    plt.bar(x - width, precision, width, label='precision')
    plt.bar(x, recall, width, label='recall')
    plt.bar(x + width, f1, width, label='f1-score')
    plt.bar(x + 2 * width, auc_scores, width, label='AUC')
    plt.xticks(x, names, rotation=30, ha='right')
    plt.ylim(0, 1.05)
    plt.ylabel('Score')
    plt.title('Perbandingan Semua Skenario')
    plt.grid(axis='y', alpha=0.3)
    plt.legend(ncol=5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scenario_comparison.png', dpi=180)
    plt.show()


def summarize_results(results: list[dict]) -> list[dict]:
    rows = []
    for result in results:
        rows.append({
            'scenario': result['scenario'],
            'model_type': result['model_type'],
            'accuracy': round(result['accuracy'], 4),
            'precision': round(result['precision'], 4),
            'recall': round(result['recall'], 4),
            'f1_score': round(result['f1_score'], 4),
            'auc': round(result['auc'], 4),
            'epochs': result.get('epochs_trained'),
            'best_val_loss': None if result.get('best_val_loss') is None else round(result['best_val_loss'], 4),
            'best_val_accuracy': None if result.get('best_val_accuracy') is None else round(result['best_val_accuracy'], 4),
            'purpose': result.get('purpose', ''),
            'config': result.get('config', {}),
        })
    return rows


summary_rows = summarize_results(scenario_results)
print(json.dumps(summary_rows, indent=2))
plot_scenario_comparison(scenario_results)

best_test_result = max(scenario_results, key=lambda item: (item['f1_score'], item['auc'], item['accuracy']))
cnn_results = [result for result in scenario_results if result['model_type'] == 'Custom CNN Conv2D from scratch']
best_cnn_result = min(cnn_results, key=lambda item: item.get('best_val_loss', np.inf))
print('Best test scenario for reporting:', best_test_result['scenario'])
print('Final selected CNN scenario by validation loss:', best_cnn_result['scenario'])
print('Final selected CNN config:', json.dumps(best_cnn_result['config'], indent=2))


## 5. Analisis Akademik

Dataset dipilih karena sesuai langsung dengan masalah klasifikasi penggunaan masker dan memiliki dua kelas yang jelas. Karakteristik dataset meliputi variasi pose, jarak wajah, background, pencahayaan, kualitas gambar, dan kemungkinan oklusi selain masker. Variasi ini membuat pipeline computer vision perlu diuji dari preprocessing hingga evaluasi akhir.

Masalah pencahayaan dan noise ditangani dengan enhancement/restoration ringan. CLAHE meningkatkan kontras lokal pada channel luminance sehingga detail area wajah dan masker dapat lebih terlihat, sedangkan Gaussian denoise ringan mengurangi noise tanpa merusak bentuk masker. Face crop bertujuan mengurangi pengaruh background, tetapi tetap memiliki risiko jika Haar Cascade gagal mendeteksi wajah pada pose ekstrem atau citra blur.

HOG digunakan sebagai baseline klasik karena menangkap distribusi orientasi tepi dan bentuk lokal. Baseline ini penting untuk menunjukkan apakah fitur manual masih kompetitif dibanding fitur yang dipelajari otomatis. CNN from scratch digunakan karena requirement melarang bobot eksternal; seluruh representasi dipelajari langsung dari dataset praktikum.

SE block digunakan sebagai attention channel ringan. Tujuannya adalah membantu model memberi bobot lebih besar pada channel fitur yang relevan tanpa menambah model eksternal. Ablation tanpa SE diperlukan untuk melihat apakah attention benar-benar membantu atau justru menambah kompleksitas.

Augmentation realistis membantu generalisasi terhadap variasi kecil seperti arah wajah, zoom, kontras, dan brightness. Augmentation ekstrem dihindari karena dapat mengubah bentuk masker atau membuat citra tidak realistis. Overfitting dianalisis dari jarak antara training dan validation accuracy/loss. Jika training accuracy tinggi tetapi validation loss naik, model mulai menghafal data. ReduceLROnPlateau, Dropout, BatchNormalization, dan EarlyStopping digunakan untuk menekan risiko tersebut.

Kekuatan sistem ini adalah pipeline lengkap, skenario jelas, dan evaluasi menyeluruh. Kelemahannya adalah Haar Cascade tidak selalu robust, tuning masih ringan, dan performa sangat bergantung pada kualitas/variasi dataset. Pengembangan lanjutan dapat mencakup cross-validation, threshold tuning berbasis validation set, analisis error per kondisi citra, balancing tambahan jika class tidak seimbang, dan face detector yang lebih stabil selama tetap sesuai aturan praktikum.

## 6. Simpan Artefak Final

Model CNN final dipilih berdasarkan validation loss terbaik dari skenario CNN yang dijalankan. Test set tetap dipakai untuk evaluasi akhir dan pelaporan, bukan untuk memilih model.

In [ ]:
best_model = trained_models[best_cnn_result['scenario']]
final_keras_path = OUTPUT_DIR / 'face_mask_custom_cnn_from_scratch_best.keras'
metrics_path = OUTPUT_DIR / 'metrics.json'
best_model.save(final_keras_path)

metrics = {
    'seed': SEED,
    'tensorflow_version': tf.__version__,
    'dataset_handle': DATASET_HANDLE,
    'dataset_root': str(dataset_root),
    'original_image_directory': str(original_image_dir),
    'class_names': CLASS_NAMES,
    'class_counts': class_counts,
    'split_ratio': {'train': TRAIN_RATIO, 'validation': VAL_RATIO, 'test': TEST_RATIO},
    'split_sizes': split_sizes,
    'split_label_counts': split_counts,
    'image_size': list(IMAGE_SIZE),
    'batch_size_default': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'eda_summary': eda_summary,
    'preprocessing_stats': preprocessing_stats,
    'deep_learning_model': 'Custom CNN Conv2D from scratch',
    'best_test_scenario_for_reporting': best_test_result['scenario'],
    'best_cnn_scenario': best_cnn_result['scenario'],
    'best_cnn_config': best_cnn_result['config'],
    'summary_rows': summary_rows,
    'scenario_results': scenario_results,
    'outputs': {
        'best_keras_model': str(final_keras_path),
        'metrics': str(metrics_path),
        'classical_models': {
            scenario_id: classical_feature_sets[scenario_id].get('model_path')
            for scenario_id in classical_feature_sets
        },
        'eda_class_distribution': str(OUTPUT_DIR / 'eda_class_distribution.png'),
        'eda_original_samples': str(OUTPUT_DIR / 'eda_original_samples.png'),
        'eda_resolution_aspect_ratio': str(OUTPUT_DIR / 'eda_resolution_aspect_ratio.png'),
        'eda_brightness_contrast': str(OUTPUT_DIR / 'eda_brightness_contrast.png'),
        'eda_noise_blur': str(OUTPUT_DIR / 'eda_noise_blur.png'),
        'eda_edge_hog_examples': str(OUTPUT_DIR / 'eda_edge_hog_examples.png'),
        'eda_pose_background_variation': str(OUTPUT_DIR / 'eda_pose_background_variation.png'),
        'eda_face_detection_coverage': str(OUTPUT_DIR / 'eda_face_detection_coverage.png'),
        'eda_face_detection_examples': str(OUTPUT_DIR / 'eda_face_detection_examples.png'),
        'eda_class_split_distribution': str(OUTPUT_DIR / 'eda_class_split_distribution.png'),
        'class_distribution': str(OUTPUT_DIR / 'class_distribution.png'),
        'preprocessing_examples': str(OUTPUT_DIR / 'preprocessing_examples.png'),
        'scenario_comparison': str(OUTPUT_DIR / 'scenario_comparison.png'),
    },
}
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

print('Saved outputs:')
for path in [
    final_keras_path,
    metrics_path,
    *[Path(path) for path in metrics['outputs']['classical_models'].values() if path],
    OUTPUT_DIR / 'eda_class_distribution.png',
    OUTPUT_DIR / 'eda_original_samples.png',
    OUTPUT_DIR / 'eda_resolution_aspect_ratio.png',
    OUTPUT_DIR / 'eda_brightness_contrast.png',
    OUTPUT_DIR / 'eda_noise_blur.png',
    OUTPUT_DIR / 'eda_edge_hog_examples.png',
    OUTPUT_DIR / 'eda_pose_background_variation.png',
    OUTPUT_DIR / 'eda_face_detection_coverage.png',
    OUTPUT_DIR / 'eda_face_detection_examples.png',
    OUTPUT_DIR / 'eda_class_split_distribution.png',
    OUTPUT_DIR / 'class_distribution.png',
    OUTPUT_DIR / 'preprocessing_examples.png',
    OUTPUT_DIR / 'scenario_comparison.png',
]:
    if path.exists():
        print('-', path)